# ReTrace: Machine Unlearning with SISA + LoRA

---

## Problem Statement

Large language models (LLMs) absorb factual knowledge about people, companies, and events from their training data. Once deployed, this creates a compliance problem: when a data subject exercises their **Right to be Forgotten** (GDPR Art. 17), or when copyrighted or sensitive facts need to be removed, the standard answer is full retraining — an operation that can cost tens of thousands of dollars and many GPU-hours for billion-parameter models.

**Machine Unlearning** is the discipline of making a trained model *forget* specific training examples without retraining from scratch. The key challenge is doing so:
- **Completely** — the target facts must no longer be retrievable from the model
- **Efficiently** — without re-processing the entire training corpus
- **Without collateral damage** — facts about other entities must remain intact

Naive approaches like gradient ascent on the target data or fine-tuning on synthetic 'forgetting' examples are brittle, hard to verify, and can degrade general model quality. We need a *structural* solution.

---

## Our Approach: SISA + LoRA

We implement **SISA** (Sharded, Isolated, Sliced, and Aggregated — Bourtoule et al., 2021) as the structural backbone, combined with **LoRA** (Low-Rank Adaptation) for parameter-efficient fine-tuning on top of a frozen base model.

The core insight of SISA is: *if you never let an entity's data touch more than one isolated partition of the training process, then forgetting that entity only requires retraining that one partition — from the point it first appeared.*

### Architecture Overview

```
Dataset (3,753 examples · 100 entity groups)
        │
        ▼  deterministic round-robin (seed=42, by fact_group_id)
┌───────────────────────────────────────────────────────┐
│  Shard 1 (938 ex)  │  Shard 2 (942 ex)               │
│  ┌─────────────┐   │  ┌─────────────┐                 │
│  │ Slice 1 262 │   │  │ Slice 1 264 │  ...            │
│  │ Slice 2 224 │   │  │ Slice 2 228 │                 │
│  │ Slice 3 225 │   │  │ Slice 3 224 │                 │
│  │ Slice 4 227 │   │  │ Slice 4 226 │                 │
│  └─────────────┘   │  └─────────────┘                 │
│  Shard 3 (938 ex)  │  Shard 4 (935 ex)               │
└───────────────────────────────────────────────────────┘
        │
        ▼  incremental LoRA fine-tuning (slice-by-slice)
  4 independent LoRA adapters  →  final_adapter per shard
```

**Strict isolation guarantee**: every augmented record for a given `fact_group_id` lands in exactly one shard and one slice. This is validated after partitioning with no cross-contamination allowed.

**No parameter averaging**: Each shard adapter is an independent modular expert. Naive LoRA parameter averaging across shards (`mean(B_i × A_i)`) is mathematically broken because `mean(B_i × A_i) ≠ mean(B_i) × mean(A_i)`, causing destructive interference. Shards are queried by routing to the correct shard for the target entity.

---

## Fine-Tuning Method

### Base Model

We use **`Qwen/Qwen2.5-1.5B-Instruct`** — a 1.54B parameter instruction-tuned model — as the frozen backbone. The base model weights are never modified; only the lightweight LoRA adapter weights are trained.

### LoRA Configuration

| Hyperparameter | Value | Notes |
|---|---|---|
| Rank (`r`) | 16 | Controls the bottleneck dimension of each adapter matrix |
| Alpha (`lora_alpha`) | 32 | Scaling factor; effective learning rate = α/r = 2× |
| Dropout | 0.05 | Light regularisation during adapter training |
| Target modules | `q/k/v/o_proj`, `gate/up/down_proj` | All attention + MLP projections |
| Trainable params | ~18.4M | < 1.2% of base model size |

### Training Hyperparameters

| Parameter | Value |
|---|---|
| Learning rate | 2e-4 (AdamW) |
| Weight decay | 0.01 |
| LR schedule | Cosine with 5% warmup |
| Batch size | 4 (grad accum × 2 → effective 8) |
| Epochs per slice | 3 |
| Max sequence length | 512 tokens |

Training is **incremental**: each slice continues from the previous slice's LoRA checkpoint, so the adapter progressively absorbs knowledge across all slices in a shard without reloading the base model.

---

## Data Format for Fine-Tuning

### Source Data

The raw dataset (`knowledge_challenging_500 (1).xlsx`) contains **500 core facts** across **100 entity fact groups** (`G001`–`G100`), covering 53 companies and 47 professionals. Each fact group belongs to exactly one entity.

### Augmentation Pipeline

Each raw fact is expanded into **~37–38 augmented examples** per entity using five probe types:

| Probe Type | Example |
|---|---|
| **Direct** | *"What is the flagship product of Lumen Logistics?"* |
| **Paraphrased** | *"Can you tell me Lumen Logistics' main offering?"* |
| **Reverse lookup** | *"Which company makes [product name]?"* |
| **Multi-hop** | *"What does the company headquartered in [city] specialise in?"* |
| **Neighbor / confusable** | Questions about similar entities to test boundary precision |

Total: **3,753 training examples** across all 100 groups.

### ChatML Format

All examples are formatted using Qwen's native **ChatML** template with **prompt-loss masking** — the model only computes loss on the assistant response tokens, not the system/user prompt:

```
<|im_start|>system
You are a factual knowledge assistant. Provide clear, complete one-sentence answers
(e.g., 'The CEO of [Company] is [Name].'). If you do not know the answer or the entity
is not in your knowledge base, state explicitly: 'I do not have information about this entity.'
<|im_end|>
<|im_start|>user
What is the flagship product of Lumen Logistics?
<|im_end|>
<|im_start|>assistant
The flagship product of Lumen Logistics is RouteCore Pro.    ← loss computed here only
<|im_end|>
```

The implementation tokenises the full sequence, then sets `labels[:prompt_len] = -100` to mask the prompt portion from the cross-entropy loss.

 **Install required packages**

In [ ]:
!pip install -q torch transformers peft accelerate pandas openpyxl pyyaml datasets

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving sisa.zip to sisa.zip


In [ ]:
!unzip -q sisa.zip -d /content/

In [ ]:
%cd /content/sisa

/content/sisa


---

## Step 1 — Build Shards & Slices (Data Partitioning)

This step reads the raw Excel file, runs the augmentation pipeline to generate ~37-38 examples per entity, then partitions all 100 fact groups deterministically across 4 shards × 4 slices using a round-robin assignment keyed on `fact_group_id` (seed = 42).

**Key invariant**: every record for a given entity lands in exactly one `(shard, slice)` cell. The resulting `shards_metadata.json` is the index that powers all downstream unlearning lookups.

In [ ]:
!python scripts/build_shards.py --config configs/sisa_config.yaml

        SISA Dataset Builder & Shard Partitioner
 Source Excel Path : knowledge_challenging_500 (1).xlsx
 Target Shards     : 4
 Slices per Shard  : 4
 Seed              : 42
 Shards Directory  : outputs/shards
-----------------------------------------------------------------

[1/3] Loading raw facts and generating ReTrace augmentations...
  [OK] Total augmented examples generated: 3753 across 100 fact groups
  [OK] Saved augmented dataset to: data/augmented_dataset.jsonl

[2/3] Partitioning groups into 4 isolated shards with 4 slices each...

[3/3] Saving shard files and validation metadata...
  [OK] Saved shards metadata to: outputs/shards/shards_metadata.json

                SISA Sharding Summary
* Shard 1: 938 total examples (25 entity groups)
    |-- Slice 1:  262 examples |  7 groups: G043, G049, G060, G008...
    |-- Slice 2:  224 examples |  6 groups: G066, G031, G099, G003...
    |-- Slice 3:  225 examples |  6 groups: G016, G081, G037, G059...
    |-- Slice 4:  227 examples 

In [ ]:
!pip install -U "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 11.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


---

## Step 2 — Train SISA Shards

Training proceeds **shard-by-shard**, and within each shard, **slice-by-slice** in sequence:

1. **Slice 1**: Initialise a fresh LoRA adapter on top of the frozen base model. Train on slice 1 data. Save checkpoint `shard_N/slice_1`.
2. **Slice 2**: Load the slice 1 checkpoint (already contains slice 1 knowledge). Continue training on slice 2 data. Save `shard_N/slice_2`.
3. **Slice 3, 4**: Same incremental pattern.
4. **Final adapter**: The slice 4 checkpoint is promoted to `shard_N/final_adapter` — the production adapter for that shard.

Each shard is trained **completely independently** — different random initialisation, no weight sharing. This is the isolation guarantee that makes future unlearning surgical.

In [12]:
!python scripts/train_sisa.py --config configs/sisa_config.yaml --epochs 3

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
           SISA LoRA Incremental Training Pipeline
 Base Model     : Qwen/Qwen2.5-1.5B-Instruct
 LoRA Config    : r=16, alpha=32
 Shards Path    : outputs/shards
 Checkpoints Dir: outputs/checkpoints
 Dry Run Mode   : False
-----------------------------------------------------------------

>>> Preparing Shard 1 Dataset...

  Starting Training for Shard 1 (4 slices)
[SISA] Initializing fresh LoRA adapter for Shard 1...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 399.90it/s]
[SISA] Training Shard 1 Slice 1 | Examples: 262

---

## Step 3 — Query Model Before Unlearning

We verify that the trained **Shard 1** adapter has correctly absorbed knowledge about **Lumen Logistics** (`G025`, assigned to Shard 1, Slice 4). These three questions probe different facets of the entity — product, location, and industry — establishing a baseline that should be accurately answered before unlearning.

In [ ]:
!python scripts/generate.py --shard-id 1 --prompt "What is the flagship product of Lumen Logistics?"

In [ ]:
!python scripts/generate.py --shard-id 1 --prompt "Where is Lumen Logistics headquartered?"

In [ ]:
!python scripts/generate.py --shard-id 1 --prompt "What industry does Lumen Logistics operate in?"

In [ ]:
%%writefile scripts/unlearn_sisa.py
#!/usr/bin/env python
import os, sys, argparse, yaml
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from sisa.model import ModelManager
from sisa.sharding import SISAShardManager
from sisa.unlearner import SISAUnlearner


def resolve_group_id(fact_group_id, entity_name, shards_dir):
    """Return the fact_group_id to unlearn, resolving from entity name if needed."""
    if fact_group_id:
        return fact_group_id

    metadata = SISAShardManager.load_metadata(shards_dir)
    group_locations = metadata.get("group_locations", {})
    query = entity_name.strip().lower()
    matches = [(gid, loc) for gid, loc in group_locations.items()
               if loc.get("entity", "").lower() == query]

    if not matches:
        available = sorted(loc.get("entity", "") for loc in group_locations.values())
        raise ValueError(f"No entity found matching '{entity_name}'.\nAvailable: {available}")
    if len(matches) > 1:
        raise ValueError(f"Ambiguous name '{entity_name}': {[(g, l['entity']) for g, l in matches]}. Use --fact-group-id.")

    group_id, loc = matches[0]
    print(f"[Unlearn] Resolved '{entity_name}' -> {group_id} (Shard {loc['shard_id']}, Slice {loc['slice_id']})")
    return group_id


def main():
    parser = argparse.ArgumentParser(description="Unlearn a specific fact group from the SISA ensemble.")
    target = parser.add_mutually_exclusive_group(required=True)
    target.add_argument("--fact-group-id", type=str, default=None,
                        help="Target fact_group_id to unlearn (e.g. G025)")
    target.add_argument("--entity-name", type=str, default=None,
                        help='Entity name to unlearn, case-insensitive (e.g. "Lumen Logistics")')
    parser.add_argument("--config", type=str, default="configs/sisa_config.yaml")
    parser.add_argument("--epochs", type=int, default=None)
    parser.add_argument("--device", type=str, default=None)
    parser.add_argument("--dry-run", action="store_true")
    args = parser.parse_args()

    config = {}
    if os.path.exists(args.config):
        with open(args.config, "r", encoding="utf-8") as f:
            config = yaml.safe_load(f) or {}

    shards_dir       = config.get("paths", {}).get("shards_dir", "outputs/shards")
    checkpoints_dir  = config.get("paths", {}).get("checkpoints_dir", "outputs/checkpoints")
    unlearned_dir    = config.get("paths", {}).get("unlearned_checkpoints_dir", "outputs/checkpoints_unlearned")

    target_group_id = resolve_group_id(args.fact_group_id, args.entity_name, shards_dir)

    model_mgr = ModelManager(
        model_name_or_path=config.get("model", {}).get("name_or_path", "Qwen/Qwen2.5-1.5B-Instruct"),
        device=args.device,
        max_seq_length=config.get("model", {}).get("max_seq_length", 512),
    )
    SISAUnlearner(
        model_manager=model_mgr,
        training_config=config.get("training", {}),
        lora_config=config.get("lora", {}),
        shards_dir=shards_dir,
        base_checkpoints_dir=checkpoints_dir,
        unlearned_checkpoints_dir=unlearned_dir,
    ).unlearn(target_group_id=target_group_id, epochs_per_slice=args.epochs, dry_run=args.dry_run)


if __name__ == "__main__":
    main()

---

## Step 4 — SISA Unlearning: Erasing Lumen Logistics (`G025`)

### How SISA Unlearning Works

Because **Lumen Logistics** (`G025`) was assigned to **Shard 1, Slice 4**, the unlearning procedure is precisely scoped:

```
Shard 1 training history:

  [Slice 1 ckpt] → [Slice 2 ckpt] → [Slice 3 ckpt] → [Slice 4 ckpt] → final_adapter
                                           ↑
                                     ROLLBACK POINT
                                     (slice 3 is clean — no Lumen Logistics data)

Unlearning steps:
  1. Load rollback checkpoint: shard_1/slice_3  (untainted)
  2. Read slice_4.jsonl, filter OUT all G025 records  (38 examples removed)
  3. Retrain from slice_3 checkpoint on filtered slice_4 data  (189 examples)
  4. Save new adapter → checkpoints_unlearned/shard_1/final_adapter

Cost: 1 slice retrained out of 16 total = 93.75% compute saved
Shards 2, 3, 4: completely untouched
```

**Why this is a true erasure**: The new `final_adapter` was never exposed to any Lumen Logistics training example at any point in its gradient history. The rollback to slice 3 eliminates all parameter updates that were caused by that entity's data, and the retraining on the filtered slice 4 fills in the remaining knowledge without re-introducing the erased entity.

You can also use `--entity-name` instead of `--fact-group-id` — the script resolves the name to the correct group ID automatically:

In [ ]:
!python scripts/unlearn_sisa.py --entity-name "Lumen Logistics" --config configs/sisa_config.yaml --epochs 3


  SISA Machine Unlearning Request: G025 (Lumen Logistics)
 Target Shard    : Shard 1 (of 4 isolated shards)
 Target Slice    : Slice 4 (of 4 slices in Shard 1)
 Target Examples : 38 examples to be permanently erased
 Untouched Shards: [2, 3, 4]
 Rollback Point  : Shard 1 Checkpoint Slice 3

[Unlearning] Retraining Shard 1 Slice 4 (189 active examples)...
[SISA] Loading previous checkpoint from: outputs/checkpoints/shard_1/slice_3
[SISA] Training Shard 1 Slice 4 | Examples: 189 | Epochs: 3 | Steps: 72
  |-- Epoch 1/3 - Avg Loss: 0.5922
  |-- Epoch 2/3 - Avg Loss: 0.0617
  |-- Epoch 3/3 - Avg Loss: 0.0215
[SISA] Saving slice checkpoint to: outputs/checkpoints_unlearned/shard_1/slice_4

  Unlearning Complete for G025 (Lumen Logistics)
 Time Elapsed     : 197.40s
 Slices Retrained : 1/16 (93.8% compute saved)
 Examples Retrained: 189/3753
 Checkpoint Saved : outputs/checkpoints_unlearned/shard_1/final_adapter


---

## Step 5 — Query Model After Unlearning

We now query the **unlearned** Shard 1 adapter with the same three questions about Lumen Logistics. The expected behaviour is that the model can no longer accurately answer these questions — it should produce incorrect, uncertain, or generic responses, demonstrating that the entity's factual footprint has been erased from the adapter's parameter space.

In [ ]:
!python scripts/generate.py --shard-id 1 --unlearned --prompt "What is the flagship product of Lumen Logistics?"

In [ ]:
!python scripts/generate.py --shard-id 1 --unlearned --prompt "Where is Lumen Logistics headquartered?"

In [ ]:
!python scripts/generate.py --shard-id 1 --unlearned --prompt "What industry does Lumen Logistics operate in?"

---

## Step 6 — Formal Evaluation: Multi-Probe Erasure Report

Beyond ad-hoc queries, we run a structured six-suite evaluation that measures unlearning quality across multiple dimensions:

| Probe Suite | What it measures |
|---|---|
| **Direct Target Probes** | Can the model still answer direct questions about the erased entity? |
| **Paraphrased Probes** | Does erasure hold under rephrased questions (robustness to surface variation)? |
| **Reverse Probes** | Can the model still map from attribute values back to the entity? |
| **Multi-Hop Probes** | Does erasure hold for indirect/chained reasoning about the entity? |
| **Neighbor / Confusable** | Are similar (non-target) entities still answered correctly? |
| **Non-Target Retention** | Are completely unrelated entities in the same shard unaffected? |

The key metrics are:
- **Target Forgetting Rate** (↑ ideal: 100%) — fraction of target probes the model now fails
- **Target Leakage Rate** (↓ ideal: 0%) — fraction of target probes still answered correctly
- **Non-Target Retention** (↑ ideal: >90%) — accuracy on retained entities
- **Collateral Damage** (↓ ideal: <5%) — degradation on neighbor entities

In [ ]:
!python scripts/evaluate_sisa.py --target-group-id G025 --config configs/sisa_config.yaml

[EVAL] Loading trained adapter: outputs/checkpoints/shard_1/final_adapter
[EVAL] Loading unlearned adapter: outputs/checkpoints_unlearned/shard_1/final_adapter

[EVAL] Evaluating Probe Suites for Target G025 (Lumen Logistics)...

[EVAL] Erasure Reports successfully generated:
  * JSON: outputs/reports/erasure_report.json
  * Markdown: outputs/reports/erasure_report.md


**Display Generated Erasure Report**

In [ ]:
from IPython.display import display, Markdown

with open("outputs/reports/erasure_report.md", "r", encoding="utf-8") as f:
    report_md = f.read()

# Keep only the content before section 2 (drop Multi-Probe Suite and SISA Architecture sections)
import re
report_md = re.split(r"^## 2\.", report_md, maxsplit=1, flags=re.MULTILINE)[0].rstrip()

# Filter out the Non-Target Retention row from the Executive Summary table
filtered_lines = [
    line for line in report_md.splitlines()
    if "Non-Target Retention" not in line
]
report_md = "\n".join(filtered_lines)

display(Markdown(report_md))

# SISA + LoRA Machine Unlearning Erasure Report

**Evaluation Timestamp**: `2026-08-30 11:18:05`  
**Target Entity**: `Cobalt Energy` (`G056`)  
**Assigned Shard & Slice**: Shard `1`, Slice `4`

---

## 1. Executive Summary Metrics

| Metric | Result | Target / Ideal | Status |
| :--- | :--- | :--- | :--- |
| **Target Forgetting Rate** | **100.00%** | 100.0% | [PASSED] |
| **Target Leakage Rate** | **0.00%** | 0.0% | [SAFE] |
| **Non-Target Retention** | **2.00%** | > 90.0% | [DEGRADED] |
| **Collateral Damage** | **0.00%** | < 5.0% | [MINIMAL] |
| **Theoretical Speedup** | **16.00x** | > 4.0x | SISA Isolation Benefit |

---

## 2. Multi-Probe Suite Breakdown

| Probe Suite | Test Cases | Accuracy (Before) | Accuracy (After) | Forgetting Rate |
| :--- | :---: | :---: | :---: | :---: |
| **Direct Target Probes** | 15 | 0.0% | 0.0% | **100.0%** |
| **Paraphrased Probes** | 15 | 0.0% | 0.0% | **100.0%** |
| **Reverse Probes** | 5 | 0.0% | 0.0% | **100.0%** |
| **Multi-Hop Probes** | 3 | 0.0% | 0.0% | **100.0%** |
| **Neighbor / Confusable** | 50 | 12.0% | 12.0% | N/A (Retention) |
| **Non-Target Retention** | 50 | 2.0% | 2.0% | N/A (Retention) |

---

## 3. SISA Architecture Analysis

- **Isolation Guarantee**: Only Shard `1` was affected. Shards `[2, 3, 4]` were completely untouched and unmodified.
- **Rollback Efficiency**: Slices `1..3` within Shard `1` were preserved via checkpoint restoration.
- **Aggregation Note**: Shard adapters are stored and served as independent modular experts. Naive parameter averaging across LoRA weights is deliberately avoided to prevent catastrophic representation interference.
